In [29]:
import requests
import random
import json
import pandas as pd
from datetime import datetime, timedelta

In [30]:
def simula_json(alert_ids):
    """
    Gera um JSON com os detalhes dos alertas usando a lista de IDs real.
    """
    types = ["tax_incident", "invoice_discrepancy", "missing_document", "regulatory_review"]
    regions = ['BR', 'LATAM', 'MEX']
    statuses = ["open", "in_progress", "resolved"]
    assignees = ["team_a", "team_b", "team_c", "external_partner"]
    impact_levels = ["low", "medium", "high", "critical"]
    sources = ["SAP", "Internal_Tracker", "Vendor_API"]
    alerts = []

    for alert_id in alert_ids:
        creation_date = datetime.now() - timedelta(days=random.randint(0, 400))
        resolution_date = creation_date + timedelta(days=random.randint(1,15))

        alert = {
            "alert_id": alert_id,
            "type_of_alert": random.choice(types),
            "status": random.choice(statuses),
            "assigned_to": random.choice(assignees),
            "creation_date": creation_date.isoformat(),
            "resolution_date": resolution_date.isoformat(),
            "impact_level": random.choice(impact_levels),
            "region": random.choice(regions),
            "source": random.choice(sources)
        }
        alerts.append(alert)

    alerts_json = json.dumps(alerts)
    return alerts_json

In [31]:
def simular_json_response_lista(num_ids=100):
    """
    Cria a estrutura de dados Python (dicionário) que simula a resposta
    da API 'compliance_alerts?status=open&limit=100'.
    """
    # Gera 100 IDs (COMP-ALERT-10000 até COMP-ALERT-10099)
    ids_simulados = [{"id": f"COMP-ALERT-{i}"} for i in range(10000, 10000 + num_ids)]
    
    # Estrutura de resposta (com 'paging' e a lista em 'results')
    response_data = {
        "paging": {
            "total": num_ids, 
            "offset": 0, 
            "limit": num_ids
        },
        "results": ids_simulados
    }
    return response_data

In [32]:
url = "https://api.mercadolibre.com/compliance_alerts?status=open&limit=100"

try:
    response = requests.get(url, timeout=5)
    response.raise_for_status()

    data = response.json()
    results = data.get("results",[])
    if not results:
        if results is None:
            raise ValueError("Formato inesperado: chave 'results' ausente")
        else:
            print("Nenhum alerta encontrado")
    alert_ids = [item.get("id") for item in results if item.get("id")]
    print("IDs lidos com sucesso")

except Exception as e:
    print("Erro: ", e)

Erro:  404 Client Error: Not Found for url: https://api.mercadolibre.com/compliance_alerts?status=open&limit=100


In [33]:
data = simular_json_response_lista()
alert_ids = [item.get("id") for item in data.get("results", [])]

In [34]:
alert_json = simula_json(alert_ids)

In [35]:
alert_json = json.loads(alert_json)
alert_json[0]

{'alert_id': 'COMP-ALERT-10000',
 'type_of_alert': 'invoice_discrepancy',
 'status': 'open',
 'assigned_to': 'team_a',
 'creation_date': '2025-01-07T21:49:52.133349',
 'resolution_date': '2025-01-19T21:49:52.133349',
 'impact_level': 'critical',
 'region': 'BR',
 'source': 'SAP'}

In [36]:
# alert_detail = []
# for alert_id in alert_ids:
#     url = f"https://api.mercadolibre.com/compliance_alerts/{alert_id}"

#     try:
#         # response = requests.get(url, timeout=5)
#         response.raise_for_status()

#         data = response.json()
#         alert_detail.append(data)

#     except Exception as e:
#         print("Erro: ", e)

In [37]:
alert_detail = []

for alert_id in alert_ids:
    match = [alert for alert in alert_json if alert['alert_id'] == alert_id]

    if match:
        alert_detail.append(match[0])
    else:
        print("ID não encontrado")

In [38]:
df = pd.DataFrame(alert_detail)
df.to_csv("../01_Datasets/alert_detail.csv")

In [39]:
cols = ['alert_id', 'type_of_alert', 'status', 'assigned_to', 'creation_date', 'resolution_date', 'impact_level']
# df = pd.DataFrame(alert_detail[cols])

In [40]:
df = pd.json_normalize(alert_detail)
df = df[cols]
df

,alert_id,type_of_alert,status,assigned_to,creation_date,resolution_date,impact_level
0,COMP-ALERT-10000,invoice_discrepancy,open,team_a,2025-01-07T21:49:52.133349,2025-01-19T21:49:52.133349,critical
1,COMP-ALERT-10001,missing_document,in_progress,external_partner,2025-03-28T21:49:52.133349,2025-04-08T21:49:52.133349,critical
2,COMP-ALERT-10002,missing_document,in_progress,team_a,2025-04-16T21:49:52.133349,2025-04-18T21:49:52.133349,low
3,COMP-ALERT-10003,tax_incident,in_progress,team_a,2025-08-09T21:49:52.133349,2025-08-11T21:49:52.133349,high
4,COMP-ALERT-10004,regulatory_review,open,external_partner,2025-09-09T21:49:52.133349,2025-09-17T21:49:52.133349,critical
...,...,...,...,...,...,...,...
95,COMP-ALERT-10095,regulatory_review,in_progress,team_a,2025-01-01T21:49:52.134347,2025-01-07T21:49:52.134347,critical
96,COMP-ALERT-10096,tax_incident,in_progress,team_c,2025-11-12T21:49:52.134347,2025-11-24T21:49:52.134347,medium
97,COMP-ALERT-10097,regulatory_review,resolved,external_partner,2024-11-17T21:49:52.134347,2024-12-02T21:49:52.134347,low
98,COMP-ALERT-10098,tax_incident,open,team_c,2025-06-05T21:49:52.134347,2025-06-07T21:49:52.134347,low
